# M2-YOLO Aggressive Re-train (Colab T4)

**조건**: M2 첫 학습이 ep15에 0.78 미달 시 옵션 2로 재시작

**변경점:**
- 모델: yolov8m (25M) → **yolov11m (20M, +3~5 mAP)**
- imgsz: 960 → **1280** (소형 객체 직격타)
- batch: 16 → **8** (1280에 맞춤)
- epochs: 25 → **60**
- patience: 10 → **20**
- close_mosaic: 15 → **30**
- lr0: 1e-3 → **5e-4 cosine** (더 안정 수렴)
- mixup 0.05 → **0.15**, copy_paste 0.3 → **0.5**

**Drive 준비:**
1. `surface.zip` (이미 있음)

**예상 시간:** 7~9시간 (T4 batch=8 + 1280)

**Resume + Drive 자동 저장 포함**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★ Drive 폴더 ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect')  # 본인 계정에 맞게
ZIP_NAME = 'surface.zip'

WORK = Path('/content/m2_train')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
print(f'zip exists? {zip_path.exists()} -> {zip_path}')
assert zip_path.exists(), f'{zip_path} 없음'

ds_dir = WORK / 'surface'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')

for split in ['train', 'val', 'test']:
    p = ds_dir / 'images' / split
    if p.exists():
        print(f'  {split}: {len(list(p.glob("*")))} images')

In [ ]:
yaml_text = f"""path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 2
names:
  0: surface_defect_wall
  1: baseboard_defect
"""
data_yaml = WORK / 'surface.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

In [ ]:
AUTOSAVE = DRIVE_DIR / 'm2_aggressive_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/runs/m2_aggressive')
PROJECT.mkdir(parents=True, exist_ok=True)

RESUME = False
resume_pt = AUTOSAVE / 'last.pt'
if resume_pt.exists():
    print(f'Resume: {resume_pt}')
    weights_dir = PROJECT / 'train' / 'weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(resume_pt, weights_dir / 'last.pt')
    if (AUTOSAVE / 'best.pt').exists():
        shutil.copy2(AUTOSAVE / 'best.pt', weights_dir / 'best.pt')
    RESUME = True
else:
    print('Fresh aggressive train.')

In [ ]:
from ultralytics import YOLO
import threading, time

stop_flag = [False]
def autosave_loop():
    while not stop_flag[0]:
        time.sleep(300)
        for src in [PROJECT/'train/weights/last.pt', PROJECT/'train/weights/best.pt']:
            if src.exists():
                try: shutil.copy2(src, AUTOSAVE / src.name)
                except: pass
        rcsv = PROJECT/'train/results.csv'
        if rcsv.exists():
            try: shutil.copy2(rcsv, AUTOSAVE / 'results.csv')
            except: pass
        print(f'[autosave] {time.strftime("%H:%M:%S")} -> {AUTOSAVE}')

t = threading.Thread(target=autosave_loop, daemon=True)
t.start()

if RESUME:
    model = YOLO(str(PROJECT / 'train/weights/last.pt'))
    train_kwargs = {'resume': True}
else:
    model = YOLO('yolo11m.pt')
    train_kwargs = {
        'data': str(data_yaml),
        'epochs': 60,
        'batch': 8,              # T4 16GB + 1280 → batch=8
        'imgsz': 1280,
        'cache': 'disk',
        'workers': 4,
        'optimizer': 'AdamW',
        'lr0': 5e-4,
        'lrf': 0.01,
        'cos_lr': True,
        'patience': 20,
        'warmup_epochs': 3,
        'close_mosaic': 30,
        'freeze': 0,
        'hsv_h': 0.015, 'hsv_s': 0.5, 'hsv_v': 0.4,
        'degrees': 5.0, 'translate': 0.1, 'scale': 0.5,
        'shear': 2.0, 'perspective': 0.001,
        'flipud': 0.0, 'fliplr': 0.5,
        'mosaic': 1.0, 'mixup': 0.15, 'copy_paste': 0.5,
        'erasing': 0.0,
        'multi_scale': 0.2,
        'save_period': 5,
        'plots': True,
        'project': str(PROJECT.parent),
        'name': PROJECT.name,
        'exist_ok': True,
    }

results = model.train(**train_kwargs)
stop_flag[0] = True
print('Train done.')

In [ ]:
best_path = PROJECT / 'train/weights/best.pt'
best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')

metrics = best_model.val(data=str(data_yaml), imgsz=1280, batch=8)
print(f'\n=== M2-aggressive 결과 ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print(f'0.9? {"YES ✅" if metrics.box.map50 >= 0.9 else "NO"}')

OUT = DRIVE_DIR / 'm2_aggressive_results'
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, OUT / 'm2_yolo_surface.onnx')
shutil.copy2(best_path, OUT / 'm2_aggressive_best.pt')
if (best_path.parent.parent / 'results.csv').exists():
    shutil.copy2(best_path.parent.parent / 'results.csv', OUT / 'results.csv')
for plot in best_path.parent.parent.glob('*.png'):
    shutil.copy2(plot, OUT / plot.name)
print('Saved:', OUT)